In [1]:
#Tujuan cell ini:
#Memungkinkan notebook Colab mengakses file dataset, skrip, atau folder proyek yang disimpan di Google Drive.
#Mempermudah read/write file, misalnya membaca CSV, menyimpan hasil preprocessing, atau menulis output analisis langsung ke Drive.
#Menjadi langkah awal sebelum mengatur path dataset (RAW_DATA_PATH) dan folder output di notebook.
#Singkatnya, ini menghubungkan Google Drive dengan Colab agar file proyek bisa dibaca/tulis secara langsung.

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Tujuan cell ini:
#Memverifikasi apakah Google Drive sudah terhubung dengan Colab.
#Mengetahui struktur folder dan file yang tersedia untuk digunakan di notebook, misalnya dataset CSV, skrip Python, atau folder proyek.
#Membantu menentukan path file yang benar untuk membaca data atau menyimpan output.
#Singkatnya, cell ini berfungsi sebagai cek isi folder Drive agar langkah berikutnya dalam notebook dapat menggunakan path file dengan tepat.

!ls "/content/drive/MyDrive"

 app-release.apk  'Colab Notebooks'   PA3KEL10


In [3]:
#Fase 1 Verifikasi Isi Folder Data Mentah.
#Tujuan cell ini sangat sederhana — memastikan file dataset benar-benar ada di folder data/raw sebelum notebook mulai bekerja.

!ls "/content/drive/MyDrive/PA3KEL10/proyek_bidat/data/raw"

'Road Accident Data.csv'


In [4]:
#Fase 2 Verifikasi Struktur Folder Proyek.
#Tujuan cell ini adalah memastikan folder proyek utama sudah ada dan melihat struktur direktorinya setelah Google Drive berhasil di-mount.

!ls "/content/drive/MyDrive/PA3KEL10/proyek_bidat"

dashboard  docs       README.md		      slides  Untitled0.ipynb
data	   notebooks  requirements_colab.txt  src     video


In [5]:
# Fase 3 Konfigurasi Path & Persiapan Folder Proyek
# Menyiapkan folder proyek dan mengecek apakah dataset tersedia di dalam Google Drive
from pathlib import Path
import os

BASE_DIR = Path('/content/drive/MyDrive/PA3KEL10/proyek_bidat')

RAW_DATA_PATH = BASE_DIR / 'data' / 'raw' / 'Road Accident Data.csv'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
OUTPUT_DIR = BASE_DIR / 'data' / 'output'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('BASE_DIR:', BASE_DIR)
print('RAW_DATA_PATH:', RAW_DATA_PATH)
print('Dataset ada:', RAW_DATA_PATH.exists())

BASE_DIR: /content/drive/MyDrive/PA3KEL10/proyek_bidat
RAW_DATA_PATH: /content/drive/MyDrive/PA3KEL10/proyek_bidat/data/raw/Road Accident Data.csv
Dataset ada: True


## **Dataset Profiling**

In [6]:
#Fase 4 Dataset Profilling
#Tujuan cell ini adalah mengenal kondisi awal dataset secara menyeluruh sebelum data diproses lebih lanjut.

import pandas as pd
import json

df_pd = pd.read_csv(RAW_DATA_PATH)

original_rows = df_pd.shape[0]
original_columns = df_pd.shape[1]
original_column_names = df_pd.columns.tolist()

# Buat salinan khusus untuk profiling agar data asli tidak berubah
df_profile = df_pd.copy()

df_profile['Accident_Severity_Clean'] = df_profile['Accident_Severity'].replace({'Fetal': 'Fatal'})
df_profile['date_parsed'] = pd.to_datetime(df_profile['Accident Date'], format='%d-%m-%Y', errors='coerce')
df_profile['hour'] = pd.to_datetime(df_profile['Time'], format='%H:%M', errors='coerce').dt.hour

profile = {
    'original_rows': int(original_rows),
    'original_columns': int(original_columns),
    'original_column_names': original_column_names,
    'columns_after_profiling': int(df_profile.shape[1]),
    'profiling_added_columns': [
        'Accident_Severity_Clean',
        'date_parsed',
        'hour'
    ],
    'duplicate_rows': int(df_pd.duplicated().sum()),
    'missing_values_original': df_pd.isna().sum().sort_values(ascending=False).astype(int).to_dict(),
    'years': sorted(df_profile['date_parsed'].dt.year.dropna().astype(int).unique().tolist()),
    'severity_counts': df_profile['Accident_Severity_Clean'].value_counts().astype(int).to_dict(),
    'top_districts': df_pd['Local_Authority_(District)'].value_counts().head(10).astype(int).to_dict(),
    'top_vehicle_types': df_pd['Vehicle_Type'].value_counts().head(10).astype(int).to_dict(),
    'total_casualties': int(df_pd['Number_of_Casualties'].sum()),
    'total_vehicles': int(df_pd['Number_of_Vehicles'].sum())
}

profile_path = OUTPUT_DIR / 'dataset_profile.json'
profile_path.write_text(json.dumps(profile, indent=2, ensure_ascii=False), encoding='utf-8')

profile

{'original_rows': 307973,
 'original_columns': 21,
 'original_column_names': ['Accident_Index',
  'Accident Date',
  'Day_of_Week',
  'Junction_Control',
  'Junction_Detail',
  'Accident_Severity',
  'Latitude',
  'Light_Conditions',
  'Local_Authority_(District)',
  'Carriageway_Hazards',
  'Longitude',
  'Number_of_Casualties',
  'Number_of_Vehicles',
  'Police_Force',
  'Road_Surface_Conditions',
  'Road_Type',
  'Speed_limit',
  'Time',
  'Urban_or_Rural_Area',
  'Weather_Conditions',
  'Vehicle_Type'],
 'columns_after_profiling': 24,
 'profiling_added_columns': ['Accident_Severity_Clean', 'date_parsed', 'hour'],
 'duplicate_rows': 0,
 'missing_values_original': {'Carriageway_Hazards': 302549,
  'Time': 17,
  'Day_of_Week': 0,
  'Accident Date': 0,
  'Accident_Index': 0,
  'Junction_Detail': 0,
  'Junction_Control': 0,
  'Light_Conditions': 0,
  'Accident_Severity': 0,
  'Local_Authority_(District)': 0,
  'Longitude': 0,
  'Number_of_Casualties': 0,
  'Latitude': 0,
  'Number_of_Ve

# **Spark Session**

In [7]:
# FASE 5A - Spark Session
# Tujuan:
# 1. Mengaktifkan Apache Spark di Google Colab
# 2. Membuat objek spark agar bisa membaca dan memproses dataset besar

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('RoadAccidentUK_BigData_Colab')
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate()
)

spark

# **Spark Preprocessing**

In [8]:
# FASE 5B - Load Data dengan Spark
# Tujuan:
# 1. Membaca dataset mentah dari Google Drive
# 2. Mengecek jumlah baris dan kolom awal
# 3. Menampilkan contoh data awal

df_raw = (
    spark.read
    .option('header', True)
    .option('inferSchema', True)
    .csv(str(RAW_DATA_PATH))
)

print("Jumlah baris awal:", df_raw.count())
print("Jumlah kolom awal:", len(df_raw.columns))
print("Daftar kolom awal:")
print(df_raw.columns)

df_raw.show(5, truncate=False)

Jumlah baris awal: 307973
Jumlah kolom awal: 21
Daftar kolom awal:
['Accident_Index', 'Accident Date', 'Day_of_Week', 'Junction_Control', 'Junction_Detail', 'Accident_Severity', 'Latitude', 'Light_Conditions', 'Local_Authority_(District)', 'Carriageway_Hazards', 'Longitude', 'Number_of_Casualties', 'Number_of_Vehicles', 'Police_Force', 'Road_Surface_Conditions', 'Road_Type', 'Speed_limit', 'Time', 'Urban_or_Rural_Area', 'Weather_Conditions', 'Vehicle_Type']
+--------------+-------------+-----------+------------------------+-----------------------+-----------------+---------+---------------------+--------------------------+-------------------+---------+--------------------+------------------+-------------------+-----------------------+------------------+-----------+-------------------+-------------------+------------------+---------------------+
|Accident_Index|Accident Date|Day_of_Week|Junction_Control        |Junction_Detail        |Accident_Severity|Latitude |Light_Conditions     |Lo

In [9]:
# FASE 5C - Normalisasi Nama Kolom
# Tujuan:
# 1. Mengubah nama kolom seperti "Accident Date" menjadi "accident_date"
# 2. Mengubah "Local_Authority_(District)" menjadi "local_authority_district"
# 3. Memudahkan pemanggilan kolom pada tahap preprocessing dan analisis

def normalize_column_name(name: str) -> str:
    return (
        name.strip()
        .lower()
        .replace(' ', '_')
        .replace('(', '')
        .replace(')', '')
        .replace('-', '_')
        .replace('/', '_')
    )

df_stage = df_raw

for old_name in df_stage.columns:
    df_stage = df_stage.withColumnRenamed(old_name, normalize_column_name(old_name))

print("Nama kolom setelah normalisasi:")
print(df_stage.columns)

df_stage.show(5, truncate=False)

Nama kolom setelah normalisasi:
['accident_index', 'accident_date', 'day_of_week', 'junction_control', 'junction_detail', 'accident_severity', 'latitude', 'light_conditions', 'local_authority_district', 'carriageway_hazards', 'longitude', 'number_of_casualties', 'number_of_vehicles', 'police_force', 'road_surface_conditions', 'road_type', 'speed_limit', 'time', 'urban_or_rural_area', 'weather_conditions', 'vehicle_type']
+--------------+-------------+-----------+------------------------+-----------------------+-----------------+---------+---------------------+------------------------+-------------------+---------+--------------------+------------------+-------------------+-----------------------+------------------+-----------+-------------------+-------------------+------------------+---------------------+
|accident_index|accident_date|day_of_week|junction_control        |junction_detail        |accident_severity|latitude |light_conditions     |local_authority_district|carriageway_haza

In [10]:
from pyspark.sql.functions import (
    col,
    count,
    sum as spark_sum,
    when,
    trim,
    lower,
    to_date
)

from pyspark.sql.types import StringType

noise_report = {}

total_rows = df_stage.count()
total_columns = len(df_stage.columns)

noise_report["total_rows"] = total_rows
noise_report["total_columns"] = total_columns

print("Import fungsi Spark berhasil.")
print("Jumlah baris:", total_rows)
print("Jumlah kolom:", total_columns)

Import fungsi Spark berhasil.
Jumlah baris: 307973
Jumlah kolom: 21


In [11]:
# FASE 5D.1 - Deteksi Missing Value
# Tujuan:
# 1. Menghitung jumlah nilai kosong pada setiap kolom
# 2. Menentukan kolom mana yang perlu ditangani pada tahap cleaning

missing_exprs = []

for c in df_stage.columns:
    missing_exprs.append(
        spark_sum(
            when(
                col(c).isNull() | (trim(col(c).cast("string")) == ""),
                1
            ).otherwise(0)
        ).alias(c)
    )

missing_df = df_stage.select(missing_exprs)

missing_pd = (
    missing_df
    .toPandas()
    .T
    .rename(columns={0: "missing_count"})
    .sort_values("missing_count", ascending=False)
)

missing_pd["missing_percentage"] = (missing_pd["missing_count"] / total_rows * 100).round(2)

display(missing_pd)

# Simpan ringkasan missing value ke noise_report
noise_report["missing_values"] = missing_pd["missing_count"].to_dict()

,missing_count,missing_percentage
time,17,0.01
carriageway_hazards,3,0.00
day_of_week,0,0.00
accident_date,0,0.00
accident_index,0,0.00
junction_detail,0,0.00
junction_control,0,0.00
light_conditions,0,0.00
accident_severity,0,0.00
local_authority_district,0,0.00


In [12]:
# FASE 5D.2 - Deteksi Nilai Placeholder pada Kolom Kategorikal
# Tujuan:
# 1. Membedakan missing value aktual dengan nilai placeholder
# 2. Mengecek jumlah nilai seperti None, Unknown, N/A, Null, dan NaN
# 3. Menjelaskan mengapa Carriageway_Hazards memiliki banyak nilai "None"

from pyspark.sql.functions import lower, trim, col, sum as spark_sum, when

placeholder_values = ["none", "unknown", "null", "n/a", "na", "nan", "-"]

placeholder_exprs = []

for c in df_stage.columns:
    placeholder_exprs.append(
        spark_sum(
            when(
                lower(trim(col(c).cast("string"))).isin(placeholder_values),
                1
            ).otherwise(0)
        ).alias(c)
    )

placeholder_df = df_stage.select(placeholder_exprs)

placeholder_pd = (
    placeholder_df
    .toPandas()
    .T
    .rename(columns={0: "placeholder_count"})
    .sort_values("placeholder_count", ascending=False)
)

placeholder_pd["placeholder_percentage"] = (
    placeholder_pd["placeholder_count"] / total_rows * 100
).round(2)

display(placeholder_pd)

# Cek khusus nilai unik pada kolom carriageway_hazards
print("Distribusi nilai pada carriageway_hazards:")
df_stage.groupBy("carriageway_hazards").count().orderBy(col("count").desc()).show(truncate=False)

,placeholder_count,placeholder_percentage
carriageway_hazards,302546,98.24
accident_date,0,0.00
day_of_week,0,0.00
junction_control,0,0.00
accident_index,0,0.00
junction_detail,0,0.00
accident_severity,0,0.00
light_conditions,0,0.00
latitude,0,0.00
local_authority_district,0,0.00


Distribusi nilai pada carriageway_hazards:
+-----------------------------------------------+------+
|carriageway_hazards                            |count |
+-----------------------------------------------+------+
|None                                           |302546|
|Other object on road                           |2243  |
|Any animal in carriageway (except ridden horse)|1620  |
|Pedestrian in carriageway - not injured        |715   |
|Previous accident                              |511   |
|Vehicle load on road                           |335   |
|NULL                                           |3     |
+-----------------------------------------------+------+



In [13]:
# FASE 5D.3 - Deteksi Duplikasi Data
# Tujuan:
# 1. Mengecek duplikasi seluruh baris
# 2. Mengecek duplikasi berdasarkan accident_index

duplicate_all_rows = total_rows - df_stage.dropDuplicates().count()

duplicate_accident_index_df = (
    df_stage
    .groupBy("accident_index")
    .count()
    .filter(col("count") > 1)
)

duplicate_accident_index_count = duplicate_accident_index_df.count()

print("Jumlah duplikasi seluruh baris:", duplicate_all_rows)
print("Jumlah accident_index yang duplikat:", duplicate_accident_index_count)

print("Contoh accident_index duplikat jika ada:")
duplicate_accident_index_df.show(10, truncate=False)

noise_report["duplicate_all_rows"] = duplicate_all_rows
noise_report["duplicate_accident_index_count"] = duplicate_accident_index_count

Jumlah duplikasi seluruh baris: 0
Jumlah accident_index yang duplikat: 0
Contoh accident_index duplikat jika ada:
+--------------+-----+
|accident_index|count|
+--------------+-----+
+--------------+-----+



In [14]:
# FASE 5D.4 - Deteksi Inkonsistensi Accident Severity
# Tujuan:
# 1. Melihat distribusi kategori severity sebelum cleaning
# 2. Mengecek apakah ada typo seperti Fetal
# 3. Mengecek kategori severity di luar Fatal, Serious, dan Slight

print("Distribusi accident_severity sebelum cleaning:")
severity_before = (
    df_stage
    .groupBy("accident_severity")
    .count()
    .orderBy(col("count").desc())
)

severity_before.show(truncate=False)

# Deteksi typo Fetal
fetal_count = (
    df_stage
    .filter(lower(col("accident_severity")) == "fetal")
    .count()
)

# Deteksi severity di luar kategori valid
valid_severity = ["fatal", "serious", "slight"]

invalid_severity_df = (
    df_stage
    .filter(~lower(col("accident_severity")).isin(valid_severity))
    .groupBy("accident_severity")
    .count()
)

invalid_severity_count = invalid_severity_df.count()

print("Jumlah typo severity 'Fetal':", fetal_count)
print("Jumlah jenis kategori severity tidak valid:", invalid_severity_count)

print("Kategori severity tidak valid jika ada:")
invalid_severity_df.show(truncate=False)

noise_report["fetal_typo_count"] = fetal_count
noise_report["invalid_severity_category_count"] = invalid_severity_count

Distribusi accident_severity sebelum cleaning:
+-----------------+------+
|accident_severity|count |
+-----------------+------+
|Slight           |263280|
|Serious          |40740 |
|Fatal            |3904  |
|Fetal            |49    |
+-----------------+------+

Jumlah typo severity 'Fetal': 49
Jumlah jenis kategori severity tidak valid: 1
Kategori severity tidak valid jika ada:
+-----------------+-----+
|accident_severity|count|
+-----------------+-----+
|Fetal            |49   |
+-----------------+-----+



In [15]:
# FASE 5D.5 - Deteksi Time dan Accident Date Tidak Valid
# Tujuan:
# 1. Mengecek time kosong atau tidak valid
# 2. Menangani kasus ketika Spark membaca kolom time sebagai timestamp
# 3. Mengecek accident_date yang tidak bisa dikonversi menjadi tanggal

from pyspark.sql.functions import col, trim, to_date, date_format

# Cek tipe data kolom time
time_dtype = dict(df_stage.dtypes).get("time")
print("Tipe data kolom time:", time_dtype)

# Jika time terbaca sebagai timestamp, ubah dulu ke format HH:mm
# Jika time masih string, cukup trim seperti biasa
if time_dtype == "timestamp":
    df_time_check = df_stage.withColumn(
        "time_text",
        date_format(col("time"), "HH:mm")
    )
else:
    df_time_check = df_stage.withColumn(
        "time_text",
        trim(col("time").cast("string"))
    )

# Deteksi time kosong atau tidak valid
invalid_time_df = (
    df_time_check
    .filter(
        col("time_text").isNull() |
        (trim(col("time_text")) == "") |
        (~col("time_text").rlike("^([01]?[0-9]|2[0-3]):[0-5][0-9]$"))
    )
)

invalid_time_count = invalid_time_df.count()

print("Jumlah time kosong atau tidak valid:", invalid_time_count)

print("Contoh data dengan time kosong atau tidak valid:")
invalid_time_df.select(
    "accident_index",
    "accident_date",
    "day_of_week",
    "time",
    "time_text"
).show(10, truncate=False)

# Cek tanggal tidak valid
invalid_date_df = (
    df_stage
    .withColumn("date_check", to_date(col("accident_date"), "dd-MM-yyyy"))
    .filter(col("date_check").isNull())
)

invalid_date_count = invalid_date_df.count()

print("Jumlah accident_date tidak valid:", invalid_date_count)

print("Contoh accident_date tidak valid jika ada:")
invalid_date_df.select(
    "accident_index",
    "accident_date",
    "date_check"
).show(10, truncate=False)

noise_report["invalid_time_count"] = invalid_time_count
noise_report["invalid_date_count"] = invalid_date_count

Tipe data kolom time: timestamp
Jumlah time kosong atau tidak valid: 17
Contoh data dengan time kosong atau tidak valid:
+--------------+-------------+-----------+----+---------+
|accident_index|accident_date|day_of_week|time|time_text|
+--------------+-------------+-----------+----+---------+
|BS0152771     |12-01-2021   |Monday     |NULL|NULL     |
|BS0152814     |22-01-2021   |Thursday   |NULL|NULL     |
|BS0152837     |27-01-2021   |Tuesday    |NULL|NULL     |
|BS0152855     |01-02-2021   |Sunday     |NULL|NULL     |
|BS0152859     |03-02-2021   |Tuesday    |NULL|NULL     |
|BS0152927     |28-02-2021   |Saturday   |NULL|NULL     |
|BS0153008     |23-03-2021   |Monday     |NULL|NULL     |
|BS0153087     |01-04-2021   |Wednesday  |NULL|NULL     |
|BS0153203     |18-05-2021   |Monday     |NULL|NULL     |
|BS0153297     |10-06-2021   |Wednesday  |NULL|NULL     |
+--------------+-------------+-----------+----+---------+
only showing top 10 rows
Jumlah accident_date tidak valid: 0
Contoh

In [16]:
# FASE 5D.6 - Deteksi Koordinat Tidak Valid
# Tujuan:
# 1. Mengecek latitude kosong
# 2. Mengecek longitude kosong
# 3. Mengecek koordinat di luar rentang kasar wilayah Inggris
# 4. Menyiapkan dasar untuk membuat coordinate_valid pada tahap cleaning

from pyspark.sql.functions import col

invalid_coordinate_df = (
    df_stage
    .filter(
        col("latitude").isNull() |
        col("longitude").isNull() |
        (~col("latitude").between(49.0, 61.0)) |
        (~col("longitude").between(-9.0, 3.0))
    )
)

invalid_coordinate_count = invalid_coordinate_df.count()

print("Jumlah koordinat tidak valid:", invalid_coordinate_count)

print("Contoh koordinat tidak valid jika ada:")
invalid_coordinate_df.select(
    "accident_index",
    "latitude",
    "longitude",
    "local_authority_district"
).show(10, truncate=False)

noise_report["invalid_coordinate_count"] = invalid_coordinate_count

Jumlah koordinat tidak valid: 0
Contoh koordinat tidak valid jika ada:
+--------------+--------+---------+------------------------+
|accident_index|latitude|longitude|local_authority_district|
+--------------+--------+---------+------------------------+
+--------------+--------+---------+------------------------+



In [17]:
# FASE 5D.7 - Deteksi Nilai Numerik Tidak Logis
# Tujuan:
# 1. Mengecek number_of_casualties negatif
# 2. Mengecek number_of_vehicles negatif
# 3. Mengecek speed_limit kurang dari atau sama dengan 0

from pyspark.sql.functions import col

invalid_numeric_df = (
    df_stage
    .filter(
        (col("number_of_casualties") < 0) |
        (col("number_of_vehicles") < 0) |
        (col("speed_limit") <= 0)
    )
)

invalid_numeric_count = invalid_numeric_df.count()

print("Jumlah nilai numerik tidak logis:", invalid_numeric_count)

print("Contoh nilai numerik tidak logis jika ada:")
invalid_numeric_df.select(
    "accident_index",
    "number_of_casualties",
    "number_of_vehicles",
    "speed_limit"
).show(10, truncate=False)

print("Statistik ringkas kolom numerik:")
df_stage.select(
    "number_of_casualties",
    "number_of_vehicles",
    "speed_limit",
    "latitude",
    "longitude"
).describe().show(truncate=False)

noise_report["invalid_numeric_count"] = invalid_numeric_count

Jumlah nilai numerik tidak logis: 0
Contoh nilai numerik tidak logis jika ada:
+--------------+--------------------+------------------+-----------+
|accident_index|number_of_casualties|number_of_vehicles|speed_limit|
+--------------+--------------------+------------------+-----------+
+--------------+--------------------+------------------+-----------+

Statistik ringkas kolom numerik:
+-------+--------------------+------------------+-----------------+------------------+------------------+
|summary|number_of_casualties|number_of_vehicles|speed_limit      |latitude          |longitude         |
+-------+--------------------+------------------+-----------------+------------------+------------------+
|count  |307973              |307973            |307973           |307973            |307973            |
|mean   |1.3568819344552931  |1.8290629373354157|38.86603695778526|52.48700471916564 |-1.368884096800689|
|stddev |0.8158569393614254  |0.7104766166622086|14.03293319589298|1.339010759797

In [18]:
# FASE 5D.8 - Deteksi Spasi Berlebih pada Data Teks
# Tujuan:
# 1. Mengecek kolom teks yang memiliki spasi berlebih
# 2. Menentukan perlunya trim pada tahap cleaning

from pyspark.sql.functions import col, trim, when, sum as spark_sum
from pyspark.sql.types import StringType

string_cols = [
    field.name for field in df_stage.schema.fields
    if isinstance(field.dataType, StringType)
]

whitespace_exprs = []

for c in string_cols:
    whitespace_exprs.append(
        spark_sum(
            when(col(c) != trim(col(c)), 1).otherwise(0)
        ).alias(c)
    )

whitespace_df = df_stage.select(whitespace_exprs)

whitespace_pd = (
    whitespace_df
    .toPandas()
    .T
    .rename(columns={0: "whitespace_count"})
    .sort_values("whitespace_count", ascending=False)
)

display(whitespace_pd)

noise_report["whitespace_issues"] = whitespace_pd["whitespace_count"].to_dict()

,whitespace_count
accident_index,0
accident_date,0
day_of_week,0
junction_control,0
junction_detail,0
accident_severity,0
light_conditions,0
local_authority_district,0
carriageway_hazards,0
police_force,0


In [19]:
# FASE 5D.9 - Ringkasan Masalah Data
# Tujuan:
# 1. Merangkum seluruh noise yang sudah ditemukan
# 2. Menjadi dasar tindakan cleaning

import pandas as pd

summary_noise = {
    "total_rows": noise_report.get("total_rows"),
    "total_columns": noise_report.get("total_columns"),
    "duplicate_all_rows": noise_report.get("duplicate_all_rows"),
    "duplicate_accident_index_count": noise_report.get("duplicate_accident_index_count"),
    "fetal_typo_count": noise_report.get("fetal_typo_count"),
    "invalid_severity_category_count": noise_report.get("invalid_severity_category_count"),
    "invalid_time_count": noise_report.get("invalid_time_count"),
    "invalid_date_count": noise_report.get("invalid_date_count"),
    "invalid_coordinate_count": noise_report.get("invalid_coordinate_count"),
    "invalid_numeric_count": noise_report.get("invalid_numeric_count"),
}

summary_noise_pd = pd.DataFrame(
    list(summary_noise.items()),
    columns=["Jenis Masalah", "Jumlah"]
)

display(summary_noise_pd)

print("Kolom dengan missing value tertinggi:")
display(missing_pd.head(10))

print("""
Kesimpulan awal:
1. Kolom time memiliki sejumlah data kosong atau tidak valid.
2. Kolom carriageway_hazards memiliki sedikit missing value aktual, sedangkan nilai None dapat dimaknai sebagai tidak ada hazard.
3. Jika terdapat typo Fetal pada accident_severity, maka akan diperbaiki menjadi Fatal.
4. Kolom teks perlu dirapikan menggunakan trim.
5. Koordinat dan nilai numerik perlu diberi flag validitas untuk kebutuhan analisis spasial dan clustering.
""")

,Jenis Masalah,Jumlah
0,total_rows,307973
1,total_columns,21
2,duplicate_all_rows,0
3,duplicate_accident_index_count,0
4,fetal_typo_count,49
5,invalid_severity_category_count,1
6,invalid_time_count,17
7,invalid_date_count,0
8,invalid_coordinate_count,0
9,invalid_numeric_count,0


Kolom dengan missing value tertinggi:


,missing_count,missing_percentage
time,17,0.01
carriageway_hazards,3,0.00
day_of_week,0,0.00
accident_date,0,0.00
accident_index,0,0.00
junction_detail,0,0.00
junction_control,0,0.00
light_conditions,0,0.00
accident_severity,0,0.00
local_authority_district,0,0.00



Kesimpulan awal:
1. Kolom time memiliki sejumlah data kosong atau tidak valid.
2. Kolom carriageway_hazards memiliki sedikit missing value aktual, sedangkan nilai None dapat dimaknai sebagai tidak ada hazard.
3. Jika terdapat typo Fetal pada accident_severity, maka akan diperbaiki menjadi Fatal.
4. Kolom teks perlu dirapikan menggunakan trim.
5. Koordinat dan nilai numerik perlu diberi flag validitas untuk kebutuhan analisis spasial dan clustering.



In [20]:
# FASE 5E - Cleaning Data Teks dan Kategorikal
# Tujuan:
# 1. Membersihkan spasi berlebih pada kolom teks
# 2. Menstandarkan kategori accident_severity
# 3. Memperbaiki typo "Fetal" menjadi "Fatal"
# 4. Mengimputasi missing value aktual pada carriageway_hazards dengan "None"
# 5. Mengimputasi missing value aktual pada kolom kategorikal lain dengan "Unknown"

from pyspark.sql.functions import col, trim, lower, when, lit
from pyspark.sql.types import StringType

# Data input berasal dari FASE 5B
df_clean_step1 = df_stage

print("Jumlah baris sebelum cleaning kategori:", df_clean_step1.count())
print("Jumlah kolom sebelum cleaning kategori:", len(df_clean_step1.columns))

# Mengambil semua kolom bertipe teks/string
string_cols = [
    field.name for field in df_clean_step1.schema.fields
    if isinstance(field.dataType, StringType)
]

print("Kolom bertipe teks yang akan dibersihkan:")
print(string_cols)

# 1. Menghapus spasi berlebih pada semua kolom teks
for c in string_cols:
    df_clean_step1 = df_clean_step1.withColumn(c, trim(col(c)))

# 2. Standarisasi accident_severity
# Fetal dianggap typo dan diperbaiki menjadi Fatal
df_clean_step1 = df_clean_step1.withColumn(
    "accident_severity",
    when(lower(trim(col("accident_severity"))) == "fetal", lit("Fatal"))
    .when(lower(trim(col("accident_severity"))) == "fatal", lit("Fatal"))
    .when(lower(trim(col("accident_severity"))) == "serious", lit("Serious"))
    .when(lower(trim(col("accident_severity"))) == "slight", lit("Slight"))
    .otherwise(lit("Unknown"))
)

# 3. Imputasi khusus carriageway_hazards
# Missing value aktual diisi dengan "None"
# None dimaknai sebagai tidak terdapat hazard atau hambatan pada jalan
df_clean_step1 = df_clean_step1.withColumn(
    "carriageway_hazards",
    when(
        col("carriageway_hazards").isNull() |
        (trim(col("carriageway_hazards").cast("string")) == ""),
        lit("None")
    ).otherwise(col("carriageway_hazards"))
)

# 4. Imputasi kolom kategorikal lain
# Jika ada data kosong aktual, maka diisi "Unknown"
category_cols_unknown = [
    "day_of_week",
    "junction_control",
    "junction_detail",
    "light_conditions",
    "local_authority_district",
    "police_force",
    "road_surface_conditions",
    "road_type",
    "urban_or_rural_area",
    "weather_conditions",
    "vehicle_type"
]

for c in category_cols_unknown:
    if c in df_clean_step1.columns:
        df_clean_step1 = df_clean_step1.withColumn(
            c,
            when(
                col(c).isNull() |
                (trim(col(c).cast("string")) == ""),
                lit("Unknown")
            ).otherwise(col(c))
        )

# 5. Cek hasil cleaning severity
print("Distribusi accident_severity setelah cleaning:")
df_clean_step1.groupBy("accident_severity").count().orderBy(col("count").desc()).show(truncate=False)

# 6. Cek hasil imputasi carriageway_hazards
print("Distribusi carriageway_hazards setelah imputasi:")
df_clean_step1.groupBy("carriageway_hazards").count().orderBy(col("count").desc()).show(truncate=False)

# 7. Cek missing value pada kolom kategorikal penting
print("Cek missing value setelah cleaning pada kolom kategorikal penting:")

check_cols = [
    "accident_severity",
    "carriageway_hazards",
    "weather_conditions",
    "road_surface_conditions",
    "vehicle_type"
]

for c in check_cols:
    missing_count = df_clean_step1.filter(
        col(c).isNull() |
        (trim(col(c).cast("string")) == "")
    ).count()

    print(f"{c}: {missing_count} missing value")

# 8. Menampilkan contoh data setelah cleaning
print("Contoh data setelah cleaning teks dan kategorikal:")
df_clean_step1.select(
    "accident_index",
    "accident_severity",
    "carriageway_hazards",
    "weather_conditions",
    "road_surface_conditions",
    "vehicle_type"
).show(10, truncate=False)

Jumlah baris sebelum cleaning kategori: 307973
Jumlah kolom sebelum cleaning kategori: 21
Kolom bertipe teks yang akan dibersihkan:
['accident_index', 'accident_date', 'day_of_week', 'junction_control', 'junction_detail', 'accident_severity', 'light_conditions', 'local_authority_district', 'carriageway_hazards', 'police_force', 'road_surface_conditions', 'road_type', 'urban_or_rural_area', 'weather_conditions', 'vehicle_type']
Distribusi accident_severity setelah cleaning:
+-----------------+------+
|accident_severity|count |
+-----------------+------+
|Slight           |263280|
|Serious          |40740 |
|Fatal            |3953  |
+-----------------+------+

Distribusi carriageway_hazards setelah imputasi:
+-----------------------------------------------+------+
|carriageway_hazards                            |count |
+-----------------------------------------------+------+
|None                                           |302549|
|Other object on road                           |2243  

In [21]:
# FASE 5F - Cleaning Data Waktu dan Tanggal
# Tujuan:
# 1. Mengubah time menjadi format HH:mm
# 2. Mengimputasi time kosong atau tidak valid
# 3. Mengubah accident_date menjadi format tanggal
# 4. Membuat fitur accident_year, accident_month, accident_day, dan hour

from pyspark.sql.functions import (
    col, trim, when, lit, to_date, year, month,
    dayofmonth, split, date_format
)

df_clean_step2 = df_clean_step1

# Cek tipe data kolom time
time_dtype_clean = dict(df_clean_step2.dtypes).get("time")
print("Tipe data kolom time sebelum cleaning:", time_dtype_clean)

# Jika time terbaca sebagai timestamp, ambil jam dan menit
if time_dtype_clean == "timestamp":
    df_clean_step2 = df_clean_step2.withColumn(
        "time_text",
        date_format(col("time"), "HH:mm")
    )
else:
    df_clean_step2 = df_clean_step2.withColumn(
        "time_text",
        trim(col("time").cast("string"))
    )

# Mengambil nilai time yang paling sering muncul dari data valid
mode_time = (
    df_clean_step2
    .filter(
        col("time_text").isNotNull() &
        (trim(col("time_text")) != "") &
        (col("time_text").rlike("^([01]?[0-9]|2[0-3]):[0-5][0-9]$"))
    )
    .groupBy("time_text")
    .count()
    .orderBy(col("count").desc())
    .first()[0]
)

print("Nilai time yang digunakan untuk imputasi:", mode_time)

# Imputasi time kosong atau tidak valid
df_clean_step2 = df_clean_step2.withColumn(
    "time",
    when(
        col("time_text").isNull() |
        (trim(col("time_text")) == "") |
        (~col("time_text").rlike("^([01]?[0-9]|2[0-3]):[0-5][0-9]$")),
        lit(mode_time)
    ).otherwise(col("time_text"))
)

# Konversi tanggal dan ekstraksi fitur waktu
df_clean_step2 = (
    df_clean_step2
    .withColumn("accident_date_parsed", to_date(col("accident_date"), "dd-MM-yyyy"))
    .withColumn("accident_year", year(col("accident_date_parsed")))
    .withColumn("accident_month", month(col("accident_date_parsed")))
    .withColumn("accident_day", dayofmonth(col("accident_date_parsed")))
    .withColumn("hour", split(col("time"), ":").getItem(0).cast("int"))
    .drop("time_text")
)

print("Cek hasil fitur waktu setelah cleaning dan imputasi:")
df_clean_step2.select(
    "accident_date",
    "accident_date_parsed",
    "accident_year",
    "accident_month",
    "accident_day",
    "time",
    "hour"
).show(10, truncate=False)

print("Jumlah time yang masih kosong:")
print(df_clean_step2.filter(col("time").isNull() | (trim(col("time")) == "")).count())

print("Jumlah hour yang masih kosong:")
print(df_clean_step2.filter(col("hour").isNull()).count())

Tipe data kolom time sebelum cleaning: timestamp
Nilai time yang digunakan untuk imputasi: 17:00
Cek hasil fitur waktu setelah cleaning dan imputasi:
+-------------+--------------------+-------------+--------------+------------+-----+----+
|accident_date|accident_date_parsed|accident_year|accident_month|accident_day|time |hour|
+-------------+--------------------+-------------+--------------+------------+-----+----+
|01-01-2021   |2021-01-01          |2021         |1             |1           |15:11|15  |
|05-01-2021   |2021-01-05          |2021         |1             |5           |10:59|10  |
|04-01-2021   |2021-01-04          |2021         |1             |4           |14:19|14  |
|05-01-2021   |2021-01-05          |2021         |1             |5           |08:10|8   |
|06-01-2021   |2021-01-06          |2021         |1             |6           |17:25|17  |
|01-01-2021   |2021-01-01          |2021         |1             |1           |11:48|11  |
|08-01-2021   |2021-01-08          |2021

In [22]:
# FASE 5G - Feature Engineering Severity dan Weekend
# Tujuan:
# 1. Membuat fitur is_weekend
# 2. Membuat fitur severity_rank
# 3. Membuat fitur high_severity_flag

from pyspark.sql.functions import col, when, lit

df_clean_step3 = (
    df_clean_step2
    .withColumn(
        "is_weekend",
        when(col("day_of_week").isin("Saturday", "Sunday"), lit(1)).otherwise(lit(0))
    )
    .withColumn(
        "severity_rank",
        when(col("accident_severity") == "Fatal", lit(3))
        .when(col("accident_severity") == "Serious", lit(2))
        .when(col("accident_severity") == "Slight", lit(1))
        .otherwise(lit(0))
    )
    .withColumn(
        "high_severity_flag",
        when(col("accident_severity").isin("Fatal", "Serious"), lit(1)).otherwise(lit(0))
    )
)

print("Cek fitur severity dan weekend:")
df_clean_step3.select(
    "day_of_week",
    "is_weekend",
    "accident_severity",
    "severity_rank",
    "high_severity_flag"
).show(20, truncate=False)

print("Distribusi weekend vs weekday:")
df_clean_step3.groupBy("is_weekend").count().show()

print("Distribusi high severity:")
df_clean_step3.groupBy("high_severity_flag").count().show()

Cek fitur severity dan weekend:
+-----------+----------+-----------------+-------------+------------------+
|day_of_week|is_weekend|accident_severity|severity_rank|high_severity_flag|
+-----------+----------+-----------------+-------------+------------------+
|Thursday   |0         |Serious          |2            |1                 |
|Monday     |0         |Serious          |2            |1                 |
|Sunday     |1         |Slight           |1            |0                 |
|Monday     |0         |Serious          |2            |1                 |
|Tuesday    |0         |Serious          |2            |1                 |
|Thursday   |0         |Slight           |1            |0                 |
|Thursday   |0         |Serious          |2            |1                 |
|Friday     |0         |Slight           |1            |0                 |
|Wednesday  |0         |Slight           |1            |0                 |
|Saturday   |1         |Slight           |1            |

In [23]:
# FASE 5H - Validasi Koordinat dan Nilai Numerik
# Tujuan:
# 1. Membuat coordinate_valid
# 2. Membuat numeric_valid
# 3. Menyiapkan data untuk analisis peta dan clustering

from pyspark.sql.functions import col, when, lit

df_clean_step4 = (
    df_clean_step3
    .withColumn(
        "coordinate_valid",
        when(
            (col("latitude").between(49.0, 61.0)) &
            (col("longitude").between(-9.0, 3.0)),
            lit(1)
        ).otherwise(lit(0))
    )
    .withColumn(
        "numeric_valid",
        when(
            (col("number_of_casualties") >= 0) &
            (col("number_of_vehicles") >= 0) &
            (col("speed_limit") > 0),
            lit(1)
        ).otherwise(lit(0))
    )
)

print("Distribusi validitas koordinat:")
df_clean_step4.groupBy("coordinate_valid").count().show()

print("Distribusi validitas numerik:")
df_clean_step4.groupBy("numeric_valid").count().show()

print("Contoh data validitas koordinat dan numerik:")
df_clean_step4.select(
    "latitude",
    "longitude",
    "coordinate_valid",
    "speed_limit",
    "number_of_vehicles",
    "number_of_casualties",
    "numeric_valid"
).show(10, truncate=False)

Distribusi validitas koordinat:
+----------------+------+
|coordinate_valid| count|
+----------------+------+
|               1|307973|
+----------------+------+

Distribusi validitas numerik:
+-------------+------+
|numeric_valid| count|
+-------------+------+
|            1|307973|
+-------------+------+

Contoh data validitas koordinat dan numerik:
+---------+---------+----------------+-----------+------------------+--------------------+-------------+
|latitude |longitude|coordinate_valid|speed_limit|number_of_vehicles|number_of_casualties|numeric_valid|
+---------+---------+----------------+-----------+------------------+--------------------+-------------+
|51.512273|-0.201349|1               |30         |2                 |1                   |1            |
|51.514399|-0.199248|1               |30         |2                 |11                  |1            |
|51.486668|-0.179599|1               |30         |2                 |1                   |1            |
|51.507804|-0.20

In [24]:
# FASE 5I - Finalisasi Data Bersih
# Tujuan:
# 1. Memilih kolom final yang relevan
# 2. Menghapus duplikasi berdasarkan accident_index
# 3. Membentuk dataframe final bernama df_clean

selected_cols = [
    "accident_index",
    "accident_date",
    "accident_date_parsed",
    "accident_year",
    "accident_month",
    "accident_day",
    "day_of_week",
    "time",
    "hour",
    "is_weekend",
    "accident_severity",
    "severity_rank",
    "high_severity_flag",
    "latitude",
    "longitude",
    "coordinate_valid",
    "numeric_valid",
    "local_authority_district",
    "police_force",
    "urban_or_rural_area",
    "junction_control",
    "junction_detail",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "carriageway_hazards",
    "road_type",
    "speed_limit",
    "vehicle_type",
    "number_of_vehicles",
    "number_of_casualties"
]

existing_cols = [c for c in selected_cols if c in df_clean_step4.columns]

df_clean = (
    df_clean_step4
    .select(*existing_cols)
    .dropDuplicates(["accident_index"])
)

print("Jumlah record final setelah preprocessing:", df_clean.count())
print("Jumlah kolom final:", len(df_clean.columns))

df_clean.printSchema()
df_clean.show(5, truncate=False)

Jumlah record final setelah preprocessing: 307973
Jumlah kolom final: 31
root
 |-- accident_index: string (nullable = true)
 |-- accident_date: string (nullable = true)
 |-- accident_date_parsed: date (nullable = true)
 |-- accident_year: integer (nullable = true)
 |-- accident_month: integer (nullable = true)
 |-- accident_day: integer (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- time: string (nullable = true)
 |-- hour: integer (nullable = true)
 |-- is_weekend: integer (nullable = false)
 |-- accident_severity: string (nullable = false)
 |-- severity_rank: integer (nullable = false)
 |-- high_severity_flag: integer (nullable = false)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- coordinate_valid: integer (nullable = false)
 |-- numeric_valid: integer (nullable = false)
 |-- local_authority_district: string (nullable = true)
 |-- police_force: string (nullable = true)
 |-- urban_or_rural_area: string (nullable = true)
 |-- 

In [25]:
# FASE 5J - Cek Kualitas Data Setelah Preprocessing
# Tujuan:
# 1. Mengecek missing value setelah cleaning
# 2. Mengecek typo Fetal sudah hilang
# 3. Mengecek time dan hour sudah tidak kosong
# 4. Mengecek fitur baru berhasil dibuat

from pyspark.sql.functions import col, trim, when, sum as spark_sum, lower

missing_exprs_after = []

for c in df_clean.columns:
    missing_exprs_after.append(
        spark_sum(
            when(
                col(c).isNull() | (trim(col(c).cast("string")) == ""),
                1
            ).otherwise(0)
        ).alias(c)
    )

missing_after_df = df_clean.select(missing_exprs_after)

missing_after_pd = (
    missing_after_df
    .toPandas()
    .T
    .rename(columns={0: "missing_count_after"})
    .sort_values("missing_count_after", ascending=False)
)

display(missing_after_pd)

print("Distribusi severity setelah preprocessing:")
df_clean.groupBy("accident_severity").count().orderBy(col("count").desc()).show()

print("Jumlah typo Fetal setelah preprocessing:")
print(df_clean.filter(lower(col("accident_severity")) == "fetal").count())

print("Jumlah time kosong setelah preprocessing:")
print(df_clean.filter(col("time").isNull() | (trim(col("time")) == "")).count())

print("Jumlah hour kosong setelah preprocessing:")
print(df_clean.filter(col("hour").isNull()).count())

print("Distribusi coordinate_valid:")
df_clean.groupBy("coordinate_valid").count().show()

print("Distribusi numeric_valid:")
df_clean.groupBy("numeric_valid").count().show()

print("Cek contoh data final:")
df_clean.select(
    "accident_index",
    "accident_date",
    "hour",
    "is_weekend",
    "accident_severity",
    "severity_rank",
    "high_severity_flag",
    "coordinate_valid",
    "numeric_valid"
).show(10, truncate=False)

,missing_count_after
accident_index,0
accident_date,0
accident_date_parsed,0
accident_year,0
accident_month,0
accident_day,0
day_of_week,0
time,0
hour,0
is_weekend,0


Distribusi severity setelah preprocessing:
+-----------------+------+
|accident_severity| count|
+-----------------+------+
|           Slight|263280|
|          Serious| 40740|
|            Fatal|  3953|
+-----------------+------+

Jumlah typo Fetal setelah preprocessing:
0
Jumlah time kosong setelah preprocessing:
0
Jumlah hour kosong setelah preprocessing:
0
Distribusi coordinate_valid:
+----------------+------+
|coordinate_valid| count|
+----------------+------+
|               1|307973|
+----------------+------+

Distribusi numeric_valid:
+-------------+------+
|numeric_valid| count|
+-------------+------+
|            1|307973|
+-------------+------+

Cek contoh data final:
+--------------+-------------+----+----------+-----------------+-------------+------------------+----------------+-------------+
|accident_index|accident_date|hour|is_weekend|accident_severity|severity_rank|high_severity_flag|coordinate_valid|numeric_valid|
+--------------+-------------+----+----------+-------

In [26]:
# FASE 6 - Simpan Data Bersih
# Tujuan:
# 1. Menyimpan data bersih dalam format Parquet
# 2. Menyimpan data bersih dalam format CSV folder Spark

PARQUET_PATH = PROCESSED_DIR / "accident_clean.parquet"
CSV_PATH = PROCESSED_DIR / "accident_clean_csv"

df_clean.write.mode("overwrite").parquet(str(PARQUET_PATH))
df_clean.coalesce(1).write.mode("overwrite").option("header", True).csv(str(CSV_PATH))

print("Output Parquet:", PARQUET_PATH)
print("Output CSV:", CSV_PATH)

Output Parquet: /content/drive/MyDrive/PA3KEL10/proyek_bidat/data/processed/accident_clean.parquet
Output CSV: /content/drive/MyDrive/PA3KEL10/proyek_bidat/data/processed/accident_clean_csv


In [27]:
# Membuat file CSV final dari output Spark part-*.csv
# Spark menyimpan CSV dalam folder part-*.csv.
# Cell ini menyalin file part tersebut menjadi accident_clean.csv agar mudah digunakan.

import shutil
from pathlib import Path

def copy_part_csv_to_named_file(folder_path, output_file):
    folder_path = Path(folder_path)
    part_files = list(folder_path.glob("part-*.csv"))

    if not part_files:
        print("Tidak ada part csv di:", folder_path)
        return

    shutil.copy(part_files[0], output_file)
    print("Saved:", output_file)

copy_part_csv_to_named_file(
    PROCESSED_DIR / "accident_clean_csv",
    PROCESSED_DIR / "accident_clean.csv"
)

Saved: /content/drive/MyDrive/PA3KEL10/proyek_bidat/data/processed/accident_clean.csv


In [28]:
# FASE 7A - Load Data Bersih untuk Analisis
# Tujuan:
# 1. Membaca data bersih hasil preprocessing
# 2. Memastikan data siap digunakan untuk analisis Spark

CLEAN_PARQUET_PATH = PROCESSED_DIR / "accident_clean.parquet"

df_analysis = spark.read.parquet(str(CLEAN_PARQUET_PATH))

print("Jumlah baris data analisis:", df_analysis.count())
print("Jumlah kolom data analisis:", len(df_analysis.columns))

df_analysis.printSchema()
df_analysis.show(5, truncate=False)

Jumlah baris data analisis: 307973
Jumlah kolom data analisis: 31
root
 |-- accident_index: string (nullable = true)
 |-- accident_date: string (nullable = true)
 |-- accident_date_parsed: date (nullable = true)
 |-- accident_year: integer (nullable = true)
 |-- accident_month: integer (nullable = true)
 |-- accident_day: integer (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- time: string (nullable = true)
 |-- hour: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- accident_severity: string (nullable = true)
 |-- severity_rank: integer (nullable = true)
 |-- high_severity_flag: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- coordinate_valid: integer (nullable = true)
 |-- numeric_valid: integer (nullable = true)
 |-- local_authority_district: string (nullable = true)
 |-- police_force: string (nullable = true)
 |-- urban_or_rural_area: string (nullable = true)
 |-- junction_cont

In [29]:
# FASE 7B - Spark Analysis
# Tujuan:
# 1. Membuat ringkasan severity
# 2. Membuat tren kecelakaan per jam
# 3. Membuat tren kecelakaan per bulan
# 4. Membandingkan weekend dan weekday
# 5. Menganalisis district, vehicle type, weather, road surface, dan speed limit
# 6. Membuat spatial grid summary untuk analisis hotspot awal

from pyspark.sql.functions import (
    col,
    count,
    sum as spark_sum,
    avg,
    round as spark_round
)

def write_single_csv(spark_df, output_name):
    output_path = OUTPUT_DIR / output_name
    spark_df.coalesce(1).write.mode("overwrite").option("header", True).csv(str(output_path))
    print("Output:", output_path)

df_analysis.cache()

# 1. Jumlah kecelakaan berdasarkan severity
severity_summary = (
    df_analysis
    .groupBy("accident_severity")
    .agg(
        count("*").alias("total_accidents"),
        spark_sum("number_of_vehicles").alias("total_vehicles"),
        spark_sum("number_of_casualties").alias("total_casualties")
    )
    .orderBy(col("total_accidents").desc())
)

write_single_csv(severity_summary, "severity_summary")
severity_summary.show(truncate=False)

# 2. Tren kecelakaan per jam
hourly_trend = (
    df_analysis
    .groupBy("hour")
    .agg(
        count("*").alias("total_accidents")
    )
    .orderBy("hour")
)

write_single_csv(hourly_trend, "hourly_trend")
hourly_trend.show(24, truncate=False)

# 3. Tren kecelakaan per bulan
monthly_trend = (
    df_analysis
    .groupBy("accident_year", "accident_month")
    .agg(
        count("*").alias("total_accidents")
    )
    .orderBy("accident_year", "accident_month")
)

write_single_csv(monthly_trend, "monthly_trend")
monthly_trend.show(truncate=False)

# 4. Weekend vs weekday
weekend_vs_weekday = (
    df_analysis
    .groupBy("is_weekend")
    .agg(
        count("*").alias("total_accidents"),
        spark_round(avg("high_severity_flag"), 4).alias("high_severity_rate")
    )
    .orderBy("is_weekend")
)

write_single_csv(weekend_vs_weekday, "weekend_vs_weekday")
weekend_vs_weekday.show(truncate=False)

# 5. District dengan kecelakaan tertinggi
district_summary = (
    df_analysis
    .groupBy("local_authority_district")
    .agg(
        count("*").alias("total_accidents"),
        spark_sum("number_of_casualties").alias("total_casualties"),
        spark_round(avg("high_severity_flag"), 4).alias("high_severity_rate")
    )
    .orderBy(col("total_accidents").desc())
)

write_single_csv(district_summary, "district_summary")
district_summary.show(10, truncate=False)

# 6. Vehicle type
vehicle_summary = (
    df_analysis
    .groupBy("vehicle_type")
    .agg(
        count("*").alias("total_accidents"),
        spark_round(avg("high_severity_flag"), 4).alias("high_severity_rate")
    )
    .orderBy(col("total_accidents").desc())
)

write_single_csv(vehicle_summary, "vehicle_summary")
vehicle_summary.show(10, truncate=False)

# 7. Weather by severity
weather_by_severity = (
    df_analysis
    .groupBy("weather_conditions", "accident_severity")
    .agg(
        count("*").alias("total_accidents")
    )
    .orderBy(col("total_accidents").desc())
)

write_single_csv(weather_by_severity, "weather_by_severity")
weather_by_severity.show(10, truncate=False)

# 8. Road surface by severity
road_surface_by_severity = (
    df_analysis
    .groupBy("road_surface_conditions", "accident_severity")
    .agg(
        count("*").alias("total_accidents")
    )
    .orderBy(col("total_accidents").desc())
)

write_single_csv(road_surface_by_severity, "road_surface_by_severity")
road_surface_by_severity.show(10, truncate=False)

# 9. Speed limit summary
speed_limit_summary = (
    df_analysis
    .groupBy("speed_limit")
    .agg(
        count("*").alias("total_accidents"),
        spark_round(avg("high_severity_flag"), 4).alias("high_severity_rate")
    )
    .orderBy("speed_limit")
)

write_single_csv(speed_limit_summary, "speed_limit_summary")
speed_limit_summary.show(truncate=False)

# 10. Spatial grid summary untuk hotspot awal
spatial_grid_summary = (
    df_analysis
    .filter(col("coordinate_valid") == 1)
    .withColumn("lat_grid", spark_round(col("latitude"), 2))
    .withColumn("lon_grid", spark_round(col("longitude"), 2))
    .groupBy("lat_grid", "lon_grid")
    .agg(
        count("*").alias("total_accidents"),
        spark_round(avg("high_severity_flag"), 4).alias("high_severity_rate")
    )
    .orderBy(col("total_accidents").desc())
)

write_single_csv(spatial_grid_summary, "spatial_grid_summary")
spatial_grid_summary.show(10, truncate=False)

Output: /content/drive/MyDrive/PA3KEL10/proyek_bidat/data/output/severity_summary
+-----------------+---------------+--------------+----------------+
|accident_severity|total_accidents|total_vehicles|total_casualties|
+-----------------+---------------+--------------+----------------+
|Slight           |263280         |487993        |351436          |
|Serious          |40740          |68359         |59312           |
|Fatal            |3953           |6950          |7135            |
+-----------------+---------------+--------------+----------------+

Output: /content/drive/MyDrive/PA3KEL10/proyek_bidat/data/output/hourly_trend
+----+---------------+
|hour|total_accidents|
+----+---------------+
|0   |4667           |
|1   |3533           |
|2   |2652           |
|3   |2311           |
|4   |1721           |
|5   |2491           |
|6   |5274           |
|7   |12560          |
|8   |22552          |
|9   |15560          |
|10  |13791          |
|11  |16141          |
|12  |18342       

In [30]:
# FASE 7C - Cek Output Analisis Spark
# Tujuan:
# 1. Mengecek apakah folder output analisis sudah terbentuk
# 2. Menampilkan daftar output hasil analisis Spark
# 3. Memastikan hasil analisis bisa digunakan pada fase berikutnya

from pathlib import Path

output_folders = [
    "severity_summary",
    "hourly_trend",
    "monthly_trend",
    "weekend_vs_weekday",
    "district_summary",
    "vehicle_summary",
    "weather_by_severity",
    "road_surface_by_severity",
    "speed_limit_summary",
    "spatial_grid_summary"
]

print("Cek folder output analisis Spark:")

for folder in output_folders:
    path = OUTPUT_DIR / folder
    print(folder, "->", path.exists())

Cek folder output analisis Spark:
severity_summary -> True
hourly_trend -> True
monthly_trend -> True
weekend_vs_weekday -> True
district_summary -> True
vehicle_summary -> True
weather_by_severity -> True
road_surface_by_severity -> True
speed_limit_summary -> True
spatial_grid_summary -> True


In [ ]:
# FASE 8 - Hotspot Clustering dengan Spark MLlib
#Tujuan cell ini adalah menemukan titik-titik konsentrasi kecelakaan (hotspot) secara otomatis menggunakan algoritma machine learning KMeans, lalu menyimpan hasilnya untuk analisis dan visualisasi dashboard.


from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.sql.functions import col, count, avg, round as spark_round

# Mengambil data yang koordinat dan numeriknya valid
cluster_df = (
    df_analysis
    .filter(col("coordinate_valid") == 1)
    .filter(col("numeric_valid") == 1)
    .filter(col("latitude").isNotNull() & col("longitude").isNotNull())
    .select(
        "accident_index",
        "latitude",
        "longitude",
        "speed_limit",
        "number_of_casualties",
        "number_of_vehicles",
        "high_severity_flag",
        "accident_severity",
        "local_authority_district"
    )
)

print("Jumlah data untuk clustering:", cluster_df.count())

# Menggabungkan fitur numerik ke dalam satu vector
assembler = VectorAssembler(
    inputCols=[
        "latitude",
        "longitude",
        "speed_limit",
        "number_of_casualties",
        "number_of_vehicles"
    ],
    outputCol="features_raw",
    handleInvalid="skip"
)

assembled = assembler.transform(cluster_df)

# Standardisasi fitur agar skala latitude, longitude, speed_limit, casualties, dan vehicles seimbang
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=True
)

scaled = scaler.fit(assembled).transform(assembled)

# Clustering dengan KMeans
kmeans = KMeans(
    k=10,
    seed=42,
    featuresCol="features",
    predictionCol="cluster_id"
)

model = kmeans.fit(scaled)
predictions = model.transform(scaled)

# Menyimpan hasil prediksi cluster per kecelakaan
hotspot_predictions_path = OUTPUT_DIR / "hotspot_predictions"

predictions.select(
    "accident_index",
    "latitude",
    "longitude",
    "cluster_id",
    "accident_severity",
    "high_severity_flag",
    "local_authority_district"
).coalesce(1).write.mode("overwrite").option("header", True).csv(str(hotspot_predictions_path))

# Membuat ringkasan tiap cluster hotspot
cluster_summary = (
    predictions
    .groupBy("cluster_id")
    .agg(
        count("*").alias("total_accidents"),
        spark_round(avg("latitude"), 6).alias("center_latitude"),
        spark_round(avg("longitude"), 6).alias("center_longitude"),
        spark_round(avg("high_severity_flag"), 4).alias("high_severity_rate")
    )
    .orderBy(col("total_accidents").desc())
)

hotspot_summary_path = OUTPUT_DIR / "hotspot_cluster_summary"

cluster_summary.coalesce(1).write.mode("overwrite").option("header", True).csv(str(hotspot_summary_path))

print("Training cost:", model.summary.trainingCost)

print("Ringkasan cluster hotspot:")
cluster_summary.show(10, truncate=False)

Jumlah data untuk clustering: 307973


In [ ]:
# FASE 9 - Export Output CSV Final
# Tujuan:
# 1. Mengambil file part-*.csv dari output Spark
# 2. Menyalinnya menjadi file CSV biasa dengan timestamp terbaru
# 3. Memudahkan file hasil analisis digunakan untuk dashboard dan database

import shutil
import os
from pathlib import Path

def copy_part_csv_to_named_file(folder_path, output_file):
    folder_path = Path(folder_path)
    output_file = Path(output_file)

    part_files = list(folder_path.glob("part-*.csv"))

    if not part_files:
        print(f"[SKIP] Tidak ada part csv di: {folder_path}")
        return False

    # Gunakan shutil.copy2 agar metadata (timestamp) ikut diperbarui
    shutil.copy2(part_files[0], output_file)

    # Perbarui timestamp file secara eksplisit ke waktu sekarang
    os.utime(output_file, None)

    file_size_mb = output_file.stat().st_size / (1024 * 1024)
    print(f"[OK] Saved: {output_file.name} ({file_size_mb:.2f} MB)")
    return True


# 1. Export data bersih final
print("=" * 60)
print("Export data bersih:")
copy_part_csv_to_named_file(
    PROCESSED_DIR / "accident_clean_csv",
    PROCESSED_DIR / "accident_clean.csv"
)

# 2. Export output analisis Spark
print("\nExport output analisis Spark:")
analysis_outputs = [
    "severity_summary",
    "hourly_trend",
    "monthly_trend",
    "weekend_vs_weekday",
    "district_summary",
    "vehicle_summary",
    "weather_by_severity",
    "road_surface_by_severity",
    "speed_limit_summary",
    "spatial_grid_summary",
    "hotspot_cluster_summary"
]

for name in analysis_outputs:
    copy_part_csv_to_named_file(
        OUTPUT_DIR / name,
        OUTPUT_DIR / f"{name}.csv"
    )

# 3. Export hotspot predictions untuk peta cluster
print("\nExport hotspot predictions:")
copy_part_csv_to_named_file(
    OUTPUT_DIR / "hotspot_predictions",
    OUTPUT_DIR / "hotspot_predictions.csv"
)

print("\nFASE 9 selesai.")

In [ ]:
# FASE 9B - Verifikasi File CSV Final
# Tujuan:
# 1. Memastikan semua file CSV berhasil dibuat
# 2. Mengecek ukuran file dan jumlah baris
# 3. Menampilkan preview isi file penting

import pandas as pd
from datetime import datetime

print("=" * 60)
print("FASE 9B - Verifikasi File CSV Final")
print("=" * 60)

final_files = {
    "accident_clean.csv":           PROCESSED_DIR / "accident_clean.csv",
    "severity_summary.csv":         OUTPUT_DIR / "severity_summary.csv",
    "hourly_trend.csv":             OUTPUT_DIR / "hourly_trend.csv",
    "monthly_trend.csv":            OUTPUT_DIR / "monthly_trend.csv",
    "weekend_vs_weekday.csv":       OUTPUT_DIR / "weekend_vs_weekday.csv",
    "district_summary.csv":         OUTPUT_DIR / "district_summary.csv",
    "vehicle_summary.csv":          OUTPUT_DIR / "vehicle_summary.csv",
    "weather_by_severity.csv":      OUTPUT_DIR / "weather_by_severity.csv",
    "road_surface_by_severity.csv": OUTPUT_DIR / "road_surface_by_severity.csv",
    "speed_limit_summary.csv":      OUTPUT_DIR / "speed_limit_summary.csv",
    "spatial_grid_summary.csv":     OUTPUT_DIR / "spatial_grid_summary.csv",
    "hotspot_cluster_summary.csv":  OUTPUT_DIR / "hotspot_cluster_summary.csv",
    "hotspot_predictions.csv":      OUTPUT_DIR / "hotspot_predictions.csv",
}

print(f"\n{'File':<35} {'Ada?':<8} {'Ukuran':>10} {'Dimodifikasi'}")
print("-" * 75)

all_ok = True
for name, path in final_files.items():
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        mtime   = datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
        print(f"{name:<35} {'✓':<8} {size_mb:>8.2f}MB  {mtime}")
    else:
        print(f"{name:<35} {'✗ TIDAK ADA':<8}")
        all_ok = False

print()
if all_ok:
    print("Semua file CSV berhasil dibuat.")
else:
    print("Ada file yang belum berhasil dibuat. Periksa kembali Fase 9.")

# Preview isi file penting
print("\nPreview accident_clean.csv (5 baris pertama):")
df_check = pd.read_csv(PROCESSED_DIR / "accident_clean.csv")
print(f"Jumlah baris: {df_check.shape[0]:,} | Jumlah kolom: {df_check.shape[1]}")
display(df_check.head())

print("\nPreview severity_summary.csv:")
display(pd.read_csv(OUTPUT_DIR / "severity_summary.csv"))

print("\nPreview hotspot_cluster_summary.csv:")
display(pd.read_csv(OUTPUT_DIR / "hotspot_cluster_summary.csv"))

In [ ]:
# FASE 10A - Persiapan Koneksi PostgreSQL Cloud
# Tujuan:
# 1. Menghubungkan Google Colab ke PostgreSQL Cloud
# 2. Mengecek apakah koneksi database berhasil
# 3. Menyiapkan engine SQLAlchemy untuk upload data

!pip install -q sqlalchemy psycopg2-binary

from sqlalchemy import create_engine, text
from getpass import getpass

POSTGRES_URL = getpass("Masukkan POSTGRES_URL PostgreSQL Cloud: ")

# Jika connection string masih diawali postgresql://,
# ubah menjadi postgresql+psycopg2:// agar sesuai dengan SQLAlchemy.
if POSTGRES_URL.startswith("postgresql://"):
    POSTGRES_URL = POSTGRES_URL.replace("postgresql://", "postgresql+psycopg2://", 1)

engine = create_engine(
    POSTGRES_URL,
    pool_pre_ping=True
)

with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print("Koneksi PostgreSQL berhasil.")
    print(result.fetchone()[0])

In [ ]:
from pathlib import Path
import os

BASE_DIR = Path('/content/drive/MyDrive/PA3KEL10/proyek_bidat')

RAW_DATA_PATH = BASE_DIR / 'data' / 'raw' / 'Road Accident Data.csv'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
OUTPUT_DIR = BASE_DIR / 'data' / 'output'

# FASE 10B - Upload Data ke PostgreSQL
# Tujuan:
# 1. Membaca file CSV final dari Google Drive
# 2. Mengirim accident_clean.csv ke PostgreSQL
# 3. Mengirim output analisis Spark ke PostgreSQL
# 4. Menyimpan hasil clustering ke PostgreSQL

import pandas as pd
from pathlib import Path

def upload_csv_to_postgres(csv_path, table_name, chunksize=2000):
    csv_path = Path(csv_path)

    if not csv_path.exists():
        print(f"File tidak ditemukan, dilewati: {csv_path}")
        return

    print("=" * 70)
    print(f"Membaca file: {csv_path}")

    df_temp = pd.read_csv(csv_path)

    print(f"Upload ke tabel: {table_name}")
    print(f"Jumlah baris: {df_temp.shape[0]}")
    print(f"Jumlah kolom: {df_temp.shape[1]}")

    df_temp.to_sql(
        table_name,
        engine,
        if_exists="replace",
        index=False,
        chunksize=chunksize,
        method="multi"
    )

    print(f"Tabel {table_name} berhasil dikirim ke PostgreSQL.")


# Upload data bersih utama
upload_csv_to_postgres(
    PROCESSED_DIR / "accident_clean.csv",
    "accident_clean",
    chunksize=1000
)

# Upload hasil analisis Spark dan clustering
tables_to_upload = [
    "severity_summary",
    "hourly_trend",
    "monthly_trend",
    "weekend_vs_weekday",
    "district_summary",
    "vehicle_summary",
    "weather_by_severity",
    "road_surface_by_severity",
    "speed_limit_summary",
    "spatial_grid_summary",
    "hotspot_cluster_summary",
    "hotspot_predictions"
]

for table_name in tables_to_upload:
    upload_csv_to_postgres(
        OUTPUT_DIR / f"{table_name}.csv",
        table_name,
        chunksize=2000
    )

print("\nFASE 10B selesai. Semua file yang tersedia berhasil diproses ke PostgreSQL.")

In [ ]:
# FASE 10C - Verifikasi Tabel PostgreSQL
from sqlalchemy import text

tables_to_check = [
    "accident_clean",
    "severity_summary",
    "hourly_trend",
    "monthly_trend",
    "weekend_vs_weekday",
    "district_summary",
    "vehicle_summary",
    "weather_by_severity",
    "road_surface_by_severity",
    "speed_limit_summary",
    "spatial_grid_summary",
    "hotspot_cluster_summary",
    "hotspot_predictions"
]

with engine.connect() as conn:
    print("Cek jumlah baris setiap tabel di PostgreSQL:\n")

    for table in tables_to_check:
        try:
            result = conn.execute(text(f'SELECT COUNT(*) FROM "{table}";'))
            row_count = result.fetchone()[0]
            print(f"{table}: {row_count} baris")
        except Exception as e:
            print(f"{table}: gagal dicek atau belum tersedia")

In [ ]:
# FASE 10D - Query Analisis SQL
# Tujuan:
# 1. Menampilkan jumlah kecelakaan berdasarkan severity
# 2. Menampilkan top 10 district dengan jumlah kecelakaan tertinggi
# 3. Menampilkan tren kecelakaan berdasarkan jam

import pandas as pd

query_severity = """
SELECT
    accident_severity,
    COUNT(*) AS total_accidents
FROM accident_clean
GROUP BY accident_severity
ORDER BY total_accidents DESC;
"""

query_top_district = """
SELECT
    local_authority_district,
    COUNT(*) AS total_accidents,
    SUM(number_of_casualties) AS total_casualties
FROM accident_clean
GROUP BY local_authority_district
ORDER BY total_accidents DESC
LIMIT 10;
"""

query_hourly = """
SELECT
    hour,
    COUNT(*) AS total_accidents
FROM accident_clean
GROUP BY hour
ORDER BY hour;
"""

print("Jumlah kecelakaan berdasarkan severity:")
display(pd.read_sql(query_severity, engine))

print("Top 10 district dengan jumlah kecelakaan tertinggi:")
display(pd.read_sql(query_top_district, engine))

print("Tren kecelakaan berdasarkan jam:")
display(pd.read_sql(query_hourly, engine))

In [ ]:
# FASE 11A - Koneksi MongoDB Atlas
# Tujuan:
# 1. Menghubungkan Google Colab ke MongoDB Atlas
# 2. Mengecek koneksi MongoDB
# 3. Menyiapkan database NoSQL untuk proyek

!pip install -q pymongo

from pymongo import MongoClient
from getpass import getpass

MONGO_URI = getpass("Masukkan MONGO_URI MongoDB Atlas: ")

client = MongoClient(MONGO_URI)

# Cek koneksi
client.admin.command("ping")

print("Koneksi MongoDB Atlas berhasil.")

MONGO_DB_NAME = "road_accidents_db"
db = client[MONGO_DB_NAME]

print("Database yang digunakan:", MONGO_DB_NAME)

In [ ]:
# ============================================================
# FASE 11B - Cross-System Query: PostgreSQL ↔ MongoDB
# Tujuan:
# 1. Menarik data analisis dari PostgreSQL
# 2. Menarik data clustering dari MongoDB
# 3. Menggabungkan hasil keduanya dalam satu DataFrame
# 4. Membuat ringkasan terpadu lintas sistem
# ============================================================

import pandas as pd
from sqlalchemy import text
from pymongo import MongoClient

print("=" * 65)
print("FASE 10E - Cross-System Query: PostgreSQL ↔ MongoDB")
print("=" * 65)

# ----------------------------------------------------------
# STEP 1: Ambil ringkasan severity dari PostgreSQL
# ----------------------------------------------------------
print("\n[1] Mengambil severity_summary dari PostgreSQL...")

query_severity_pg = """
SELECT
    accident_severity,
    COUNT(*)                          AS total_accidents,
    SUM(number_of_casualties)         AS total_casualties,
    ROUND(AVG(high_severity_flag), 4) AS high_severity_rate,
    ROUND(AVG(speed_limit), 2)        AS avg_speed_limit
FROM accident_clean
GROUP BY accident_severity
ORDER BY total_accidents DESC;
"""

df_severity_pg = pd.read_sql(query_severity_pg, engine)
print("PostgreSQL → severity_summary:")
display(df_severity_pg)

# ----------------------------------------------------------
# STEP 2: Ambil ringkasan cluster dari MongoDB
# ----------------------------------------------------------
print("\n[2] Mengambil hotspot_cluster_summary dari MongoDB...")

mongo_cursor = db["hotspot_cluster_summary"].find(
    {},
    {"_id": 0, "cluster_id": 1, "total_accidents": 1,
     "center_latitude": 1, "center_longitude": 1, "high_severity_rate": 1}
)

df_cluster_mongo = pd.DataFrame(list(mongo_cursor))
print("MongoDB → hotspot_cluster_summary:")
display(df_cluster_mongo)

# ----------------------------------------------------------
# STEP 3: Ambil distribusi severity per district dari PostgreSQL
# ----------------------------------------------------------
print("\n[3] Mengambil top district per severity dari PostgreSQL...")

query_district_severity = """
SELECT
    local_authority_district,
    accident_severity,
    COUNT(*) AS total_accidents
FROM accident_clean
WHERE accident_severity IN ('Fatal', 'Serious')
GROUP BY local_authority_district, accident_severity
ORDER BY total_accidents DESC
LIMIT 20;
"""

df_district_severity_pg = pd.read_sql(query_district_severity, engine)
display(df_district_severity_pg)

# ----------------------------------------------------------
# STEP 4: Ambil dokumen detail kecelakaan fatal dari MongoDB
# ----------------------------------------------------------
print("\n[4] Mengambil sample kecelakaan Fatal dari MongoDB...")

mongo_fatal_cursor = db["accident_documents"].find(
    {"severity.label": "Fatal"},
    {
        "_id": 0,
        "accident_index": 1,
        "severity.label": 1,
        "severity.rank": 1,
        "location.district": 1,
        "time_feature.hour": 1,
        "time_feature.is_weekend": 1,
        "road_condition.speed_limit": 1,
        "road_condition.weather_conditions": 1,
        "casualty.number_of_casualties": 1
    }
).limit(10)

df_fatal_mongo = pd.DataFrame(list(mongo_fatal_cursor))

# Flatten nested columns
if not df_fatal_mongo.empty:
    df_fatal_mongo["severity_label"]    = df_fatal_mongo["severity"].apply(lambda x: x.get("label") if isinstance(x, dict) else None)
    df_fatal_mongo["district"]          = df_fatal_mongo["location"].apply(lambda x: x.get("district") if isinstance(x, dict) else None)
    df_fatal_mongo["hour"]              = df_fatal_mongo["time_feature"].apply(lambda x: x.get("hour") if isinstance(x, dict) else None)
    df_fatal_mongo["is_weekend"]        = df_fatal_mongo["time_feature"].apply(lambda x: x.get("is_weekend") if isinstance(x, dict) else None)
    df_fatal_mongo["speed_limit"]       = df_fatal_mongo["road_condition"].apply(lambda x: x.get("speed_limit") if isinstance(x, dict) else None)
    df_fatal_mongo["weather"]           = df_fatal_mongo["road_condition"].apply(lambda x: x.get("weather_conditions") if isinstance(x, dict) else None)
    df_fatal_mongo["num_casualties"]    = df_fatal_mongo["casualty"].apply(lambda x: x.get("number_of_casualties") if isinstance(x, dict) else None)

    df_fatal_flat = df_fatal_mongo[[
        "accident_index", "severity_label", "district",
        "hour", "is_weekend", "speed_limit", "weather", "num_casualties"
    ]]
    print("MongoDB → sample fatal accidents (flattened):")
    display(df_fatal_flat)

# ----------------------------------------------------------
# STEP 5: Cross-join — Gabungkan district fatal (PostgreSQL)
#         dengan cluster hotspot (MongoDB)
# ----------------------------------------------------------
print("\n[5] Cross-System Join: district fatal (PG) ↔ cluster hotspot (MongoDB)...")

# Ambil semua district dengan kecelakaan Fatal dari PostgreSQL
query_fatal_district = """
SELECT
    local_authority_district,
    COUNT(*)                          AS fatal_count,
    ROUND(AVG(speed_limit), 2)        AS avg_speed_limit,
    ROUND(AVG(number_of_casualties), 3) AS avg_casualties
FROM accident_clean
WHERE accident_severity = 'Fatal'
GROUP BY local_authority_district
ORDER BY fatal_count DESC
LIMIT 20;
"""

df_fatal_pg = pd.read_sql(query_fatal_district, engine)

# Ambil data cluster dari MongoDB yang high_severity_rate tinggi
mongo_high_risk = db["hotspot_cluster_summary"].find(
    {"high_severity_rate": {"$gte": 0.2}},
    {"_id": 0, "cluster_id": 1, "total_accidents": 1,
     "center_latitude": 1, "center_longitude": 1, "high_severity_rate": 1}
)
df_high_risk_mongo = pd.DataFrame(list(mongo_high_risk))

print("District dengan fatal terbanyak (PostgreSQL):")
display(df_fatal_pg.head(10))

print("\nCluster high-risk (MongoDB, high_severity_rate ≥ 0.20):")
display(df_high_risk_mongo)

# Gabungkan: tambahkan kolom sumber sistem
df_fatal_pg["data_source"]      = "PostgreSQL"
df_high_risk_mongo["data_source"] = "MongoDB"

# Cross-system summary: berapa cluster high-risk vs total fatal district
cross_summary = {
    "total_fatal_districts_pg":        int(df_fatal_pg.shape[0]),
    "total_fatal_accidents_top20":     int(df_fatal_pg["fatal_count"].sum()),
    "total_high_risk_clusters_mongo":  int(df_high_risk_mongo.shape[0]) if not df_high_risk_mongo.empty else 0,
    "avg_speed_fatal_districts":       round(float(df_fatal_pg["avg_speed_limit"].mean()), 2),
}

print("\nRingkasan Cross-System:")
display(pd.DataFrame([cross_summary]).T.rename(columns={0: "Nilai"}))

# ----------------------------------------------------------
# STEP 6: Query MongoDB — kecelakaan jam malam fatal
#         vs data jam dari PostgreSQL
# ----------------------------------------------------------
print("\n[6] Perbandingan jam kecelakaan fatal: PostgreSQL vs MongoDB...")

# PostgreSQL: distribusi jam kecelakaan fatal
query_fatal_hour_pg = """
SELECT
    hour,
    COUNT(*) AS fatal_count_pg
FROM accident_clean
WHERE accident_severity = 'Fatal'
GROUP BY hour
ORDER BY hour;
"""
df_fatal_hour_pg = pd.read_sql(query_fatal_hour_pg, engine)

# MongoDB: distribusi jam kecelakaan fatal
mongo_hour_pipeline = [
    {"$match": {"severity.label": "Fatal"}},
    {"$group": {"_id": "$time_feature.hour", "fatal_count_mongo": {"$sum": 1}}},
    {"$sort": {"_id": 1}}
]
mongo_hour_result = list(db["accident_documents"].aggregate(mongo_hour_pipeline))
df_fatal_hour_mongo = pd.DataFrame(mongo_hour_result).rename(columns={"_id": "hour"})

# Merge hasil keduanya
df_hour_cross = pd.merge(df_fatal_hour_pg, df_fatal_hour_mongo, on="hour", how="outer").sort_values("hour")
df_hour_cross["selisih"] = df_hour_cross["fatal_count_pg"] - df_hour_cross["fatal_count_mongo"]

print("Perbandingan jumlah fatal per jam (PostgreSQL vs MongoDB):")
display(df_hour_cross)
print("\nKeterangan: selisih = 0 berarti data kedua sistem konsisten.")

# ----------------------------------------------------------
# STEP 7: Simpan hasil cross-system query ke CSV
# ----------------------------------------------------------
print("\n[7] Menyimpan hasil cross-system query...")

cross_output_path = OUTPUT_DIR / "cross_system_query_results.csv"
df_hour_cross.to_csv(cross_output_path, index=False)
print(f"Disimpan ke: {cross_output_path}")

# Upload ke PostgreSQL sebagai tabel audit
df_hour_cross.to_sql(
    "cross_system_audit_hour",
    engine,
    if_exists="replace",
    index=False
)
print("Tabel cross_system_audit_hour berhasil dibuat di PostgreSQL.")

# Upload ke MongoDB sebagai collection audit
db["cross_system_audit_hour"].drop()
db["cross_system_audit_hour"].insert_many(
    df_hour_cross.where(pd.notnull(df_hour_cross), None).to_dict(orient="records")
)
print(f"Collection cross_system_audit_hour berhasil dibuat di MongoDB.")
print(f"Dokumen tersimpan: {db['cross_system_audit_hour'].count_documents({})}")

print("\n" + "=" * 65)
print("FASE 11B selesai. Cross-system query berhasil dijalankan.")
print("=" * 65)

In [ ]:
# FASE 11C - Setup Path dan Cek File CSV
# Tujuan:
# 1. Mengaktifkan kembali path folder proyek
# 2. Mengecek file accident_clean.csv
# 3. Mengecek file hasil clustering

from pathlib import Path

BASE_DIR = Path('/content/drive/MyDrive/PA3KEL10/proyek_bidat')

PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
OUTPUT_DIR = BASE_DIR / 'data' / 'output'

print("BASE_DIR:", BASE_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

print("\nCek file penting:")
print("accident_clean.csv ada:", (PROCESSED_DIR / "accident_clean.csv").exists())
print("hotspot_cluster_summary.csv ada:", (OUTPUT_DIR / "hotspot_cluster_summary.csv").exists())
print("hotspot_predictions.csv ada:", (OUTPUT_DIR / "hotspot_predictions.csv").exists())

In [ ]:
# FASE 11D - Upload Data Bersih ke MongoDB
# Tujuan:
# 1. Membaca accident_clean.csv secara bertahap
# 2. Mengubah data menjadi dokumen nested
# 3. Menyimpan data ke collection accident_documents
# 4. Membuat index agar query lebih cepat

import pandas as pd
import numpy as np

collection = db["accident_documents"]

# Hapus collection lama agar data tidak dobel saat cell dijalankan ulang
collection.drop()

def clean_value(value):
    if pd.isna(value):
        return None

    if isinstance(value, np.generic):
        return value.item()

    return value

def row_to_document(row):
    return {
        "accident_index": clean_value(row.get("accident_index")),

        "severity": {
            "label": clean_value(row.get("accident_severity")),
            "rank": clean_value(row.get("severity_rank")),
            "high_severity_flag": clean_value(row.get("high_severity_flag"))
        },

        "location": {
            "latitude": clean_value(row.get("latitude")),
            "longitude": clean_value(row.get("longitude")),
            "district": clean_value(row.get("local_authority_district")),
            "police_force": clean_value(row.get("police_force")),
            "urban_or_rural_area": clean_value(row.get("urban_or_rural_area")),
            "coordinate_valid": clean_value(row.get("coordinate_valid"))
        },

        "time_feature": {
            "accident_date": clean_value(row.get("accident_date")),
            "accident_date_parsed": clean_value(row.get("accident_date_parsed")),
            "year": clean_value(row.get("accident_year")),
            "month": clean_value(row.get("accident_month")),
            "day": clean_value(row.get("accident_day")),
            "day_of_week": clean_value(row.get("day_of_week")),
            "time": clean_value(row.get("time")),
            "hour": clean_value(row.get("hour")),
            "is_weekend": clean_value(row.get("is_weekend"))
        },

        "road_condition": {
            "road_type": clean_value(row.get("road_type")),
            "speed_limit": clean_value(row.get("speed_limit")),
            "junction_control": clean_value(row.get("junction_control")),
            "junction_detail": clean_value(row.get("junction_detail")),
            "light_conditions": clean_value(row.get("light_conditions")),
            "weather_conditions": clean_value(row.get("weather_conditions")),
            "road_surface_conditions": clean_value(row.get("road_surface_conditions")),
            "carriageway_hazards": clean_value(row.get("carriageway_hazards"))
        },

        "vehicle": {
            "vehicle_type": clean_value(row.get("vehicle_type")),
            "number_of_vehicles": clean_value(row.get("number_of_vehicles"))
        },

        "casualty": {
            "number_of_casualties": clean_value(row.get("number_of_casualties"))
        },

        "data_quality": {
            "numeric_valid": clean_value(row.get("numeric_valid"))
        }
    }

accident_clean_path = PROCESSED_DIR / "accident_clean.csv"

batch_size = 5000
total_inserted = 0

for chunk in pd.read_csv(accident_clean_path, chunksize=batch_size):
    docs = [row_to_document(row) for _, row in chunk.iterrows()]

    if docs:
        collection.insert_many(docs, ordered=False)
        total_inserted += len(docs)
        print("Inserted:", total_inserted)

# Membuat index untuk mempercepat pencarian
collection.create_index("accident_index", unique=True)
collection.create_index("severity.label")
collection.create_index("location.district")
collection.create_index("time_feature.hour")
collection.create_index("location.coordinate_valid")

print("\nUpload accident_clean ke MongoDB selesai.")
print("Total dokumen:", collection.count_documents({}))

In [ ]:
# FASE 11E - Upload Hasil Clustering ke MongoDB
# Tujuan:
# 1. Mengirim hotspot_cluster_summary.csv ke MongoDB
# 2. Mengirim hotspot_predictions.csv ke MongoDB
# 3. Menyimpan hasil clustering dalam collection terpisah

def upload_csv_to_mongo(csv_path, collection_name):
    csv_path = Path(csv_path)

    if not csv_path.exists():
        print(f"File tidak ditemukan, dilewati: {csv_path}")
        return

    target_collection = db[collection_name]
    target_collection.drop()

    df_temp = pd.read_csv(csv_path)
    docs = df_temp.where(pd.notnull(df_temp), None).to_dict(orient="records")

    if docs:
        target_collection.insert_many(docs, ordered=False)

    print(f"Collection {collection_name} berhasil dibuat.")
    print(f"Jumlah dokumen: {target_collection.count_documents({})}\n")


upload_csv_to_mongo(
    OUTPUT_DIR / "hotspot_cluster_summary.csv",
    "hotspot_cluster_summary"
)

upload_csv_to_mongo(
    OUTPUT_DIR / "hotspot_predictions.csv",
    "hotspot_predictions"
)

In [ ]:
# FASE 11F - Verifikasi MongoDB Atlas
# Tujuan:
# 1. Mengecek jumlah dokumen setiap collection
# 2. Menampilkan contoh dokumen
# 3. Memastikan integrasi NoSQL berhasil

collections_to_check = [
    "accident_documents",
    "hotspot_cluster_summary",
    "hotspot_predictions"
]

for col_name in collections_to_check:
    col = db[col_name]
    print(f"{col_name}: {col.count_documents({})} dokumen")

print("\nContoh satu dokumen accident_documents:")
sample_doc = db["accident_documents"].find_one(
    {},
    {
        "_id": 0,
        "accident_index": 1,
        "severity": 1,
        "location": 1,
        "time_feature": 1
    }
)

sample_doc

In [ ]:
# Simpan PySpark DataFrame ke folder CSV
df_clean.write.csv('/content/accident_clean', header=True, mode='overwrite')

In [ ]:
!ls /content/

In [ ]:
!pip install streamlit plotly pyngrok

In [ ]:
from google.colab import files

# Upload file Python (.py) untuk Streamlit
uploaded = files.upload()

In [ ]:
from google.colab import files

# Upload file CSV dataset bersih
uploaded = files.upload()

In [ ]:
!ls /content/

In [ ]:
# FASE 12 - Jalankan Dashboard Streamlit via Ngrok
# Tujuan: Mendapatkan link publik untuk membuka dashboard di browser

# ── LANGKAH 1: ISI AUTHTOKEN NGROK KAMU DI SINI ─────────────────

NGROK_AUTH_TOKEN = "3E1dQWKbsUotIkAzorq78R6YI3c_2KJzgp9o7uEBWtcvc48N8"

# ── LANGKAH 2: Salin file dashboard ke /content ──────────────────
import shutil
from pathlib import Path

DASHBOARD_SRC = Path("/content/drive/MyDrive/PA3KEL10/proyek_bidat/dashboard/dashboard_streamlit_file.py")
DASHBOARD_DST = Path("/content/dashboard_streamlit.py")

if DASHBOARD_SRC.exists():
    shutil.copy2(DASHBOARD_SRC, DASHBOARD_DST)
    print(f"✓ Dashboard disalin ke {DASHBOARD_DST}")
else:
    # Jika file belum ada di Drive, tulis langsung
    print("File dashboard tidak ditemukan di Drive.")
    print("Pastikan kamu sudah upload dashboard_streamlit.py ke folder proyek di Drive.")

# ── LANGKAH 3: Jalankan Streamlit di background ──────────────────
import subprocess, time, os

port = 8501

# Matikan proses streamlit lama jika ada
os.system("pkill -f streamlit")
time.sleep(1)

proc = subprocess.Popen(
    ["streamlit", "run", str(DASHBOARD_DST),
     "--server.port", str(port),
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
print(f"✓ Streamlit berjalan di port {port} (PID: {proc.pid})")
time.sleep(4)

# ── LANGKAH 4: Buat tunnel ngrok ─────────────────────────────────
from pyngrok import ngrok, conf

# Set authtoken
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Tutup tunnel lama jika ada
ngrok.kill()
time.sleep(1)

# Buka tunnel baru
tunnel = ngrok.connect(port, "http")
public_url = tunnel.public_url

print("\n" + "="*55)
print("🚦 DASHBOARD SIAP DIAKSES!")
print("="*55)
print(f"\n  🔗 Link: {public_url}\n")
print("  Klik link di atas, lalu buka di Chrome.")
print("  Jangan matikan cell ini selama ingin menggunakan dashboard.")
print("="*55)